# Appendix 3. pandas 이해를 위한 NumPy 기초

## 1. Goal

pandas의 `Series`와 `DataFrame`은 NumPy의 배열, dtype, axis, vectorized 연산을 바탕으로 동작합니다. NumPy가 주로 **shape와 위치**를 기준으로 계산한다면, pandas는 **Index label, 열별 dtype, 결측값 처리 방식**까지 함께 관리합니다.

이 notebook에서는 NumPy API를 폭넓게 소개하지 않습니다. 대신 pandas 코드를 이해하는 데 꼭 필요한 NumPy 개념을 고르고, `ndarray`와 pandas 객체를 나란히 비교합니다. 연산 과정에서 label이 언제 유지되고 언제 사라지는지도 함께 확인합니다.

## 2. Setup

프로젝트 lock은 NumPy 2.2.6과 pandas 2.3.3을 사용합니다.

예제에는 값이 고정된 합성 ICU 데이터를 사용합니다. 외부 파일은 읽지 않습니다. 코드를 실행할 때는 결과값만 보지 말고 연산 전후의 객체 타입, shape, dtype, index, columns도 함께 확인하세요.

In [ ]:
import numpy as np
import pandas as pd

print({"numpy": np.__version__, "pandas": pd.__version__})


## 3. Steps

각 `###` 절은 pandas를 이해하는 데 필요한 NumPy 개념을 묶어서 설명합니다. `####` 절에서는 같은 개념이 NumPy와 pandas에서 어떻게 다르게 동작하는지 코드로 비교합니다.

### 3-1. ndarray와 pandas 객체 모델

#### 3-1-1. 하나의 dtype을 가진 ndarray

`ndarray`는 하나의 dtype으로 구성된 N차원 배열입니다. 같은 axis에 놓인 원소 수가 일정하므로 2차원 배열에서는 모든 row의 길이가 같습니다. `ndim`은 axis의 수, `shape`는 각 axis의 길이, `size`는 전체 원소 수를 나타냅니다. 2차원 배열은 DataFrame의 값 부분과 비슷하지만 row와 column을 설명하는 label은 없습니다.

In [ ]:
measurement_array = np.array(
    [
        [78.0, 1.8],
        [92.0, 2.6],
        [85.0, 3.1],
        [84.0, 2.2],
    ],
    dtype=np.float64,
)
print(
    {
        "type": type(measurement_array).__name__,
        "ndim": measurement_array.ndim,
        "shape": measurement_array.shape,
        "size": measurement_array.size,
        "dtype": str(measurement_array.dtype),
    }
)
measurement_array


#### 3-1-2. Series의 dtype과 DataFrame의 열별 dtypes

Series는 하나의 dtype과 Index를 가집니다. DataFrame은 같은 row Index를 공유하는 여러 Series를 column 방향으로 묶은 객체입니다. 따라서 DataFrame은 column마다 서로 다른 dtype을 가질 수 있습니다. NumPy 배열에서는 `dtype` 하나를 확인하지만, DataFrame에서는 `dtypes`로 각 column의 dtype을 확인합니다.

In [ ]:
patient_index = pd.Index(
    ["P101", "P102", "P103", "P104"],
    name="patient_id",
)
clinical_frame = pd.DataFrame(
    {
        "heart_rate": pd.Series(
            [78.0, 92.0, 85.0, 84.0],
            index=patient_index,
            dtype="Float64",
        ),
        "measurement_count": pd.Series(
            [4, 3, pd.NA, 5],
            index=patient_index,
            dtype="Int64",
        ),
        "is_complete": pd.Series(
            [True, False, False, True],
            index=patient_index,
            dtype="boolean",
        ),
    }
)
heart_rate_series = clinical_frame["heart_rate"]

print(
    {
        "series_ndim": heart_rate_series.ndim,
        "series_shape": heart_rate_series.shape,
        "series_dtype": str(heart_rate_series.dtype),
        "frame_ndim": clinical_frame.ndim,
        "frame_shape": clinical_frame.shape,
    }
)
clinical_frame.dtypes


### 3-2. dtype와 결측값

#### 3-2-1. NumPy dtype 추론과 공통 dtype 승격

NumPy 배열에는 하나의 dtype만 사용할 수 있습니다. 서로 다른 수치 dtype을 한 배열에 넣으면 NumPy는 모든 값을 표현할 수 있는 공통 dtype을 선택합니다. 이 과정을 dtype promotion이라고 합니다. `astype`으로 dtype을 직접 바꿀 때는 정밀도와 결측값 표현이 유지되는지 확인해야 합니다.

In [ ]:
integer_values = np.array([1, 2, 3], dtype=np.int32)
floating_values = np.array([1.5, 2.5, 3.5], dtype=np.float64)
combined_values = np.concatenate([integer_values, floating_values])
compact_values = floating_values.astype(np.float32)

print(
    {
        "integer_dtype": str(integer_values.dtype),
        "floating_dtype": str(floating_values.dtype),
        "combined_dtype": str(combined_values.dtype),
        "compact_dtype": str(compact_values.dtype),
        "result_type": str(
            np.result_type(integer_values.dtype, floating_values.dtype)
        ),
    }
)


#### 3-2-2. np.nan과 pandas nullable dtype

일반적인 NumPy 정수 dtype에는 결측값을 넣을 수 없습니다. 정수와 `np.nan`을 한 배열에 넣으면 floating dtype이 됩니다. pandas도 dtype을 따로 지정하지 않은 Series에 정수와 `None`을 함께 넣으면 float로 추론할 수 있습니다.

pandas의 `Int64`, `Float64`, `boolean`과 같은 nullable dtype은 `pd.NA`를 사용합니다. 덕분에 결측값이 있어도 정수나 boolean이라는 column의 의미를 유지할 수 있습니다. 결측 여부는 `pd.NA`와 직접 비교하지 말고 `isna`로 확인합니다.

In [ ]:
numpy_with_missing = np.array([1.0, np.nan, 3.0])
legacy_integer_series = pd.Series([1, None, 3])
nullable_integer_series = pd.Series([1, pd.NA, 3], dtype="Int64")
nullable_boolean_series = pd.Series([True, pd.NA, False], dtype="boolean")

print(
    {
        "numpy_dtype": str(numpy_with_missing.dtype),
        "legacy_pandas_dtype": str(legacy_integer_series.dtype),
        "nullable_integer_dtype": str(nullable_integer_series.dtype),
        "nullable_boolean_dtype": str(nullable_boolean_series.dtype),
        "nullable_missing_count": int(nullable_integer_series.isna().sum()),
    }
)


#### 3-2-3. datetime64, timedelta64와 pandas 시간 객체

NumPy의 `datetime64`는 특정 시점을, `timedelta64`는 두 시점 사이의 시간 차이를 표현합니다. pandas는 이 dtype을 바탕으로 `Timestamp`, `DatetimeIndex`, `TimedeltaIndex`를 제공하고 시간대와 빈도를 다루는 API도 지원합니다. datetime 계열의 결측값인 `NaT`는 `np.isnat` 또는 pandas의 `isna`로 확인합니다.

In [ ]:
event_times = np.array(
    [
        "2026-01-01T00:00",
        "2026-01-01T00:30",
        "2026-01-01T01:30",
    ],
    dtype="datetime64[m]",
)
elapsed_times = np.diff(event_times)
event_series = pd.Series(
    event_times,
    index=patient_index[:3],
    name="observed_at",
)
event_index = pd.DatetimeIndex(event_times, name="observed_at")
time_with_missing = np.array(
    ["2026-01-01T00:00", "NaT"],
    dtype="datetime64[m]",
)

print(
    {
        "numpy_time_dtype": str(event_times.dtype),
        "elapsed_dtype": str(elapsed_times.dtype),
        "series_dtype": str(event_series.dtype),
        "index_type": type(event_index).__name__,
        "nat_count": int(np.isnat(time_with_missing).sum()),
    }
)


### 3-3. 배열의 shape와 axis

#### 3-3-1. DataFrame 열 순서를 고정해 ndarray 만들기

NumPy 배열에는 column label이 없습니다. 따라서 DataFrame을 배열로 바꾸면 column 순서가 각 값의 의미를 결정합니다. 먼저 사용할 columns와 순서를 명시한 뒤 `to_numpy`를 호출하고 결과 shape를 확인하세요. `reshape`는 원소의 배치만 바꾸며 기존 label의 의미까지 기억하지는 않습니다.

In [ ]:
analysis_frame = pd.DataFrame(
    {
        "heart_rate": [78.0, 92.0, 85.0, 84.0],
        "lactate": [1.8, 2.6, 3.1, 2.2],
    },
    index=patient_index,
)
analysis_columns = ["heart_rate", "lactate"]
numeric_matrix = analysis_frame[analysis_columns].to_numpy(
    dtype=np.float64,
    copy=True,
)
reshaped_values = numeric_matrix.reshape(2, 4)

print(
    {
        "frame_shape": analysis_frame.shape,
        "matrix_shape": numeric_matrix.shape,
        "reshaped_shape": reshaped_values.shape,
        "column_order": analysis_columns,
    }
)


#### 3-3-2. NumPy와 pandas의 axis reduction 비교

shape가 `(rows, columns)`인 배열에서 `axis=0`으로 평균을 구하면 모든 row를 따라 계산하므로 column별 평균이 남습니다. `axis=1`로 계산하면 각 row 안의 column을 평균하므로 row별 결과가 남습니다. pandas도 같은 axis 번호를 사용하지만 결과에 해당 label을 붙여 줍니다. NumPy의 결과 shape와 pandas 결과의 Index를 함께 비교해 보세요.

In [ ]:
numpy_column_means = numeric_matrix.mean(axis=0)
numpy_row_means = numeric_matrix.mean(axis=1)
pandas_column_means = analysis_frame.mean(axis=0)
pandas_row_means = analysis_frame.mean(axis=1)

print(
    {
        "numpy_axis_0_shape": numpy_column_means.shape,
        "numpy_axis_1_shape": numpy_row_means.shape,
        "pandas_axis_0_index": pandas_column_means.index.tolist(),
        "pandas_axis_1_index": pandas_row_means.index.tolist(),
    }
)
pd.DataFrame(
    {
        "numpy": numpy_column_means,
        "pandas": pandas_column_means.to_numpy(),
    },
    index=analysis_frame.columns,
)


#### 3-3-3. keepdims와 newaxis로 계산 shape 유지하기

`keepdims=True`를 지정하면 평균을 계산한 axis가 사라지지 않고 길이 1로 남습니다. 그러면 다음 broadcasting에서 값이 어느 방향으로 확장되는지 shape만 보고도 알 수 있습니다. 1차원 배열을 row별 값으로 사용하려면 `[:, np.newaxis]`를 이용해 shape를 `(rows, 1)`로 만듭니다. pandas에서는 label을 가진 Series와 명시적인 `axis` 인자로 같은 의도를 표현할 수 있습니다.

In [ ]:
column_means_kept = numeric_matrix.mean(axis=0, keepdims=True)
centered_matrix = numeric_matrix - column_means_kept

patient_baselines_array = np.array([78.0, 90.0, 84.0, 82.0])
patient_baseline_columns = patient_baselines_array[:, np.newaxis]

print(
    {
        "mean_shape": column_means_kept.shape,
        "centered_shape": centered_matrix.shape,
        "baseline_shape": patient_baseline_columns.shape,
    }
)


### 3-4. 위치·label 선택과 copy

#### 3-4-1. ndarray 위치와 pandas iloc·loc

NumPy에서는 각 axis의 정수 위치를 쉼표로 구분해 값을 선택합니다. pandas의 `iloc`도 정수 위치를 사용하지만 `loc`는 Index와 column label을 사용합니다. 두 방식으로 같은 값을 얻을 수 있더라도 선택 기준은 다릅니다. EDA 코드를 읽을 때는 위치로 고른 것인지 label로 고른 것인지 먼저 확인하세요.

In [ ]:
numpy_first_heart_rate = numeric_matrix[0, 0]
pandas_iloc_heart_rate = analysis_frame.iloc[0, 0]
pandas_loc_heart_rate = analysis_frame.loc["P101", "heart_rate"]

print(
    {
        "numpy_position": numpy_first_heart_rate,
        "pandas_iloc": pandas_iloc_heart_rate,
        "pandas_loc": pandas_loc_heart_rate,
    }
)


#### 3-4-2. boolean mask와 결과 label

NumPy의 boolean indexing은 mask에서 `True`인 위치의 값을 골라 ndarray로 반환합니다. pandas의 boolean mask는 Series이므로 각 조건에 Index label이 붙어 있습니다. 선택한 결과에도 원래 label이 유지됩니다. pandas에서 mask를 사용할 때는 대상 객체와 mask의 Index가 서로 맞는지 확인하세요.

In [ ]:
numpy_high_hr_mask = numeric_matrix[:, 0] >= 85.0
numpy_high_hr_rows = numeric_matrix[numpy_high_hr_mask]

pandas_high_hr_mask = analysis_frame["heart_rate"].ge(85.0)
pandas_high_hr_rows = analysis_frame.loc[pandas_high_hr_mask]

print(
    {
        "numpy_mask_type": type(numpy_high_hr_mask).__name__,
        "numpy_result_shape": numpy_high_hr_rows.shape,
        "pandas_mask_type": type(pandas_high_hr_mask).__name__,
        "pandas_result_index": pandas_high_hr_rows.index.tolist(),
    }
)


#### 3-4-3. NumPy view와 pandas의 명시적 copy

NumPy의 기본 slice는 원본과 메모리를 공유하는 view일 수 있습니다. 이 view를 수정하면 원본 배열도 바뀔 수 있습니다. pandas는 dtype과 선택 방식에 따라 내부 동작이 달라지므로 NumPy의 view 규칙을 그대로 적용해서는 안 됩니다. 원본과 분리해 수정할 pandas subset은 `.loc[...].copy()`처럼 copy 의도를 코드에 명시하세요.

In [ ]:
source_array = np.array([10, 20, 30, 40])
slice_view = source_array[1:3]
slice_copy = source_array[1:3].copy()
slice_view[0] = 200

high_hr_copy = analysis_frame.loc[pandas_high_hr_mask].copy()
high_hr_copy.loc[:, "heart_rate"] = high_hr_copy["heart_rate"] - 5.0

print(
    {
        "view_shares_memory": np.shares_memory(source_array, slice_view),
        "copy_shares_memory": np.shares_memory(source_array, slice_copy),
        "numpy_source": source_array.tolist(),
        "pandas_original_P102": analysis_frame.loc["P102", "heart_rate"],
        "pandas_copy_P102": high_hr_copy.loc["P102", "heart_rate"],
    }
)


### 3-5. NumPy broadcasting과 pandas label alignment

#### 3-5-1. NumPy는 오른쪽 axis부터 shape를 비교한다

NumPy는 broadcasting 가능 여부를 판단할 때 두 배열의 shape를 오른쪽 axis부터 비교합니다. 대응하는 길이가 같거나 둘 중 하나가 1이면 계산할 수 있습니다. shape가 `(2,)`인 feature offset은 shape `(4, 2)`인 배열의 각 row에 위치 순서대로 적용됩니다. NumPy에는 label이 없으므로 두 offset이 어떤 feature를 뜻하는지는 알 수 없습니다.

In [ ]:
feature_offsets_array = np.array([80.0, 2.0])
numpy_centered = numeric_matrix - feature_offsets_array
broadcast_shape = np.broadcast_shapes(
    numeric_matrix.shape,
    feature_offsets_array.shape,
)

print(
    {
        "matrix_shape": numeric_matrix.shape,
        "offset_shape": feature_offsets_array.shape,
        "broadcast_shape": broadcast_shape,
    }
)
numpy_centered


#### 3-5-2. pandas는 Series의 label을 DataFrame columns에 맞춘다

DataFrame과 Series를 계산하면 pandas는 Series의 Index를 DataFrame의 columns와 맞춘 뒤 연산합니다. 아래 offset Series는 column 순서와 반대로 작성되어 있지만 label이 같기 때문에 올바른 column에 적용됩니다. NumPy 배열로 바꾸면 label이 사라지므로 먼저 DataFrame의 columns 순서로 `reindex`해야 합니다.

In [ ]:
feature_offsets = pd.Series(
    [2.0, 80.0],
    index=["lactate", "heart_rate"],
    name="feature_offset",
)
pandas_centered = analysis_frame - feature_offsets

wrong_positional_offsets = feature_offsets.to_numpy()
ordered_offsets = feature_offsets.reindex(analysis_frame.columns).to_numpy()
wrong_positional_result = numeric_matrix - wrong_positional_offsets
ordered_numpy_result = numeric_matrix - ordered_offsets

print(
    {
        "series_order": feature_offsets.index.tolist(),
        "frame_order": analysis_frame.columns.tolist(),
        "wrong_first_row": wrong_positional_result[0].tolist(),
        "ordered_first_row": ordered_numpy_result[0].tolist(),
        "pandas_first_row": pandas_centered.iloc[0].tolist(),
    }
)


#### 3-5-3. pandas sub에서 row label 정렬하기

NumPy에서 row별 값을 빼려면 `[:, np.newaxis]`로 column axis를 하나 추가해야 합니다. pandas에서는 `sub(..., axis="index")`를 사용해 Series의 label을 DataFrame의 row Index에 맞출 수 있습니다. Series의 순서가 달라도 label이 같으면 각 patient에 알맞은 값이 적용됩니다.

In [ ]:
heart_rate_frame = analysis_frame[["heart_rate"]]
patient_baselines = pd.Series(
    [82.0, 78.0, 84.0, 90.0],
    index=["P104", "P101", "P103", "P102"],
    name="baseline",
)
pandas_patient_change = heart_rate_frame.sub(
    patient_baselines,
    axis="index",
)

ordered_baselines = patient_baselines.reindex(heart_rate_frame.index).to_numpy()
numpy_patient_change = (
    heart_rate_frame.to_numpy() - ordered_baselines[:, np.newaxis]
)

print(
    {
        "pandas_P102": pandas_patient_change.loc["P102", "heart_rate"],
        "numpy_P102": numpy_patient_change[1, 0],
    }
)


### 3-6. Vectorized 연산과 조건부 선택

#### 3-6-1. NumPy ufunc에 Series 전달하기

`np.sqrt`, `np.log1p`와 같은 ufunc는 배열의 모든 원소에 같은 연산을 적용합니다. 많은 NumPy ufunc에 Series를 전달하면 결과도 Series로 반환되므로 Index와 name이 유지됩니다. 다만 모든 함수가 같은 방식으로 동작한다고 가정하지 말고 반환 타입을 직접 확인하세요.

In [ ]:
lactate_series = analysis_frame["lactate"]
sqrt_lactate = np.sqrt(lactate_series)
log1p_lactate = np.log1p(lactate_series)

print(
    {
        "sqrt_type": type(sqrt_lactate).__name__,
        "sqrt_name": sqrt_lactate.name,
        "sqrt_index": sqrt_lactate.index.tolist(),
        "log1p_dtype": str(log1p_lactate.dtype),
    }
)


#### 3-6-2. np.where와 Series.where의 반환 차이

`np.where`는 조건에 따라 값을 선택하고 ndarray를 반환합니다. 이 과정에서 pandas의 Index와 name은 결과에 포함되지 않습니다. `Series.where`는 조건에 맞는 값을 선택하면서 Series의 Index와 name을 유지합니다. 이후에 label을 기준으로 데이터를 결합해야 한다면 pandas 메서드를 사용하는 편이 안전합니다.

In [ ]:
high_hr_condition = analysis_frame["heart_rate"].ge(85.0)
numpy_selected_hr = np.where(
    high_hr_condition,
    analysis_frame["heart_rate"],
    np.nan,
)
pandas_selected_hr = analysis_frame["heart_rate"].where(
    high_hr_condition
)

print(
    {
        "numpy_type": type(numpy_selected_hr).__name__,
        "pandas_type": type(pandas_selected_hr).__name__,
        "pandas_index": pandas_selected_hr.index.tolist(),
        "pandas_name": pandas_selected_hr.name,
    }
)


#### 3-6-3. np.isnan·isfinite와 pandas isna

`np.isnan`은 NaN을 찾고 `np.isfinite`는 NaN과 양·음의 무한대를 제외한 값인지 확인합니다. pandas의 `isna`는 `np.nan`, `pd.NA`, `NaT`처럼 dtype마다 다른 결측값을 같은 방식으로 확인할 수 있습니다. 여러 dtype이 섞인 DataFrame의 결측값은 pandas의 `isna`로 검사하세요.

In [ ]:
quality_array = np.array([1.0, np.nan, np.inf, -np.inf, 3.0])
finite_mask = np.isfinite(quality_array)
nan_mask = np.isnan(quality_array)
frame_missing = clinical_frame.isna()

print(
    {
        "numpy_nan_count": int(nan_mask.sum()),
        "numpy_non_finite_count": int((~finite_mask).sum()),
        "pandas_missing_by_column": frame_missing.sum().to_dict(),
    }
)


### 3-7. pandas와 NumPy 변환 경계

#### 3-7-1. ndarray에서 label을 명시해 DataFrame 만들기

ndarray만 보아서는 각 axis가 무엇을 뜻하는지 알기 어렵습니다. DataFrame을 만들 때 `index`와 `columns`를 지정하면 각 row와 column의 의미를 명시할 수 있습니다. 배열의 shape와 label의 개수가 맞지 않으면 DataFrame을 만드는 단계에서 오류가 발생합니다.

In [ ]:
reconstructed_frame = pd.DataFrame(
    numeric_matrix,
    index=analysis_frame.index,
    columns=analysis_frame.columns,
)
print(
    {
        "shape": reconstructed_frame.shape,
        "index": reconstructed_frame.index.tolist(),
        "columns": reconstructed_frame.columns.tolist(),
    }
)
reconstructed_frame


#### 3-7-2. DataFrame.to_numpy에서 바뀌는 dtype과 label

`DataFrame.to_numpy`는 모든 column의 값을 담을 수 있는 공통 NumPy dtype을 선택합니다. 결과는 ndarray이므로 DataFrame의 Index와 columns는 포함되지 않습니다. 여러 extension dtype이나 문자열이 섞여 있으면 object dtype이 되거나 값 변환이 일어날 수 있습니다. 또한 `copy=False`가 copy하지 않는다는 뜻은 아니므로 메모리 공유를 전제로 코드를 작성하면 안 됩니다.

In [ ]:
mixed_array = clinical_frame.to_numpy()
numeric_nullable_array = clinical_frame[
    ["heart_rate", "measurement_count"]
].to_numpy(
    dtype=np.float64,
    na_value=np.nan,
    copy=True,
)

print(
    {
        "mixed_dtype": str(mixed_array.dtype),
        "mixed_shape": mixed_array.shape,
        "numeric_dtype": str(numeric_nullable_array.dtype),
        "numeric_missing_count": int(np.isnan(numeric_nullable_array).sum()),
    }
)


#### 3-7-3. Series.array와 Series.to_numpy 구분하기

`Series.array`는 Series를 저장하는 pandas의 ExtensionArray를 반환하므로 nullable dtype 정보가 유지됩니다. 실제 NumPy ndarray가 필요할 때는 `Series.to_numpy`를 사용합니다. 이때 extension dtype은 다른 dtype으로 변환될 수 있고 copy가 발생할 수도 있습니다. 일반적인 pandas 분석에서는 동작이 모호한 `.values`보다 목적이 분명한 두 API를 사용하세요.

In [ ]:
stored_array = nullable_integer_series.array
converted_array = nullable_integer_series.to_numpy(
    dtype=np.float64,
    na_value=np.nan,
)

print(
    {
        "stored_type": type(stored_array).__name__,
        "stored_dtype": str(stored_array.dtype),
        "converted_type": type(converted_array).__name__,
        "converted_dtype": str(converted_array.dtype),
        "converted_values": converted_array.tolist(),
    }
)


## 4. Checks

지금까지 살펴본 NumPy의 shape, dtype, 위치 연산이 pandas의 label, 열별 dtype, nullable dtype과 어떻게 연결되는지 확인합니다. assertion이 실패하면 두 객체의 `type`, `shape`, `dtype` 또는 `dtypes`, `index`, `columns`를 나란히 출력해 차이를 찾으세요.

In [ ]:
assert measurement_array.shape == (4, 2)
assert measurement_array.dtype == np.dtype("float64")
assert heart_rate_series.shape == (4,)
assert str(nullable_integer_series.dtype) == "Int64"
assert str(event_times.dtype) == "datetime64[m]"
assert numeric_matrix.shape == analysis_frame.shape
assert numpy_column_means.shape == (2,)
assert numpy_row_means.shape == (4,)
assert np.allclose(
    numpy_column_means,
    pandas_column_means.to_numpy(),
)
assert np.allclose(
    numpy_row_means,
    pandas_row_means.to_numpy(),
)
assert column_means_kept.shape == (1, 2)
assert np.shares_memory(source_array, slice_view)
assert not np.shares_memory(source_array, slice_copy)
assert pandas_high_hr_rows.index.tolist() == ["P102", "P103"]
assert np.allclose(
    ordered_numpy_result,
    pandas_centered.to_numpy(),
)
assert pandas_patient_change.loc["P102", "heart_rate"] == 2.0
assert isinstance(sqrt_lactate, pd.Series)
assert isinstance(numpy_selected_hr, np.ndarray)
assert isinstance(pandas_selected_hr, pd.Series)
assert mixed_array.dtype == np.dtype("O")
assert type(stored_array).__name__ == "IntegerArray"
assert np.isnan(converted_array[1])
print("All appendix checks passed.")


## 5. Next Steps

- NumPy는 shape와 위치를 기준으로 계산하고, pandas는 label과 열별 dtype까지 함께 고려합니다.
- DataFrame을 배열로 바꾸기 전에 column 순서를 고정하세요. 배열을 다시 DataFrame으로 만들 때는 index와 columns를 명시합니다.
- `axis=0`과 `axis=1`을 사용할 때는 결과 shape와 함께 pandas 결과에 남는 Index도 확인합니다.
- NumPy broadcasting과 pandas label alignment는 서로 다른 규칙입니다. row별 pandas 연산에는 `axis="index"`를 명시합니다.
- nullable dtype 정보는 `Series.array`에서 유지되지만 `to_numpy`에서는 다른 dtype으로 바뀔 수 있습니다.
- 조건 선택 후에도 label이 필요하면 `Series.where`를 사용합니다. label이 없는 ndarray가 필요한 경우에만 `np.where`를 사용하세요.
- 다음 notebook에서는 pandas plotting이 반환하는 객체를 이해하기 위해 Matplotlib의 Figure와 Axes를 학습합니다.

## 6. References

이 notebook의 설명과 예제는 NumPy 2.2와 pandas 2.3.3 공식 문서를 기준으로 작성했습니다.

- [NumPy absolute basics](https://numpy.org/doc/2.2/user/absolute_beginners.html)
- [NumPy data type objects](https://numpy.org/doc/2.2/reference/arrays.dtypes.html)
- [NumPy indexing](https://numpy.org/doc/2.2/user/basics.indexing.html)
- [NumPy broadcasting](https://numpy.org/doc/2.2/user/basics.broadcasting.html)
- [NumPy copies and views](https://numpy.org/doc/2.2/user/basics.copies.html)
- [pandas DataFrame.to_numpy](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.to_numpy.html)
- [pandas Series.array](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.Series.array.html)
- [pandas nullable integer dtype](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/integer_na.html)